In [ ]:
# Notebook para registrar productos mediante escáner o código y guardarlos en MongoDB
En este notebook mostramos un flujo de ejemplo para validar el código de producto y guardar el registro en MongoDB.
</VSCode.Cell>
<VSCode.Cell language="python">
# 1. Importar librerías
import pymongo
from pymongo import MongoClient
from bson.objectid import ObjectId

# Para pruebas locales, usamos input() como ejemplo de entrada manual o desde escáner integrado.
</VSCode.Cell>
<VSCode.Cell language="python">
# 2. Configurar conexión a MongoDB
MONGO_URI = "mongodb://localhost:27017"  # Ajusta a tu URI real de MongoDB si es necesario
DB_NAME = "MiniMarketInnova"
COLLECTION_NAME = "productos"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
productos_collection = db[COLLECTION_NAME]
print("Conexión a MongoDB establecida.")
</VSCode.Cell>
<VSCode.Cell language="python">
# 3. Definir función de validación de producto
# Esta función recibe un código y determina si el producto es válido.
# Aquí se puede usar una llamada a una API externa o una búsqueda en otra colección.

VALID_BARCODE_PREFIXES = ["780", "770", "770"]


def validar_producto_por_codigo(codigo: str) -> bool:
    codigo = codigo.strip()
    if not codigo.isdigit() or len(codigo) < 8:
        return False
    # Ejemplo simple: aceptamos códigos que empiecen con prefijos válidos.
    return any(codigo.startswith(prefijo) for prefijo in VALID_BARCODE_PREFIXES)
</VSCode.Cell>
<VSCode.Cell language="python">
# 4. Registrar producto desde escáner o código
# Simulamos la entrada del escáner con input. En la app real, esto vendrá del detector de códigos.

codigo_producto = input("Ingresa el código de barras o el código del producto: ").strip()
producto_valido = validar_producto_por_codigo(codigo_producto)

if producto_valido:
    producto_a_registrar = {
        "codigo": codigo_producto,
        "nombre": "Producto escaneado",
        "descripcion": "Registro desde escáner o entrada de código.",
        "creadoEn": pymongo.datetime.datetime.utcnow()
    }
else:
    producto_a_registrar = None

producto_valido, producto_a_registrar
</VSCode.Cell>
<VSCode.Cell language="python">
# 5. Guardar producto en la base de datos
if producto_valido and producto_a_registrar is not None:
    resultado = productos_collection.insert_one(producto_a_registrar)
    print(f"Producto guardado con _id: {resultado.inserted_id}")
else:
    print("No se guardó el producto porque no es válido o el código no corresponde a un producto.")
</VSCode.Cell>
<VSCode.Cell language="python">
# 6. Mostrar confirmación de producto válido o no válido
if producto_valido:
    print(f"El producto con código {codigo_producto} fue validado y registrado correctamente.")
else:
    print(f"El código {codigo_producto} no corresponde a un producto válido. Verifica el código o usa otro escaneo.")


In [1]:
import re
from pathlib import Path

root = Path(r"c:\Users\Nicolas\Documents\GitHub\MiniMarketInnova")

def replace_in_file(path, pattern, replacement, flags=0):
    text = path.read_text(encoding='utf-8')
    new_text, count = re.subn(pattern, replacement, text, flags=flags)
    if count == 0:
        raise ValueError(f"No matches for pattern in {path}")
    path.write_text(new_text, encoding='utf-8')
    print(f"Patched {path} ({count} replacements)")

escaner_path = root / 'src' / 'componentes' / 'EscanerZXing.tsx'
inventario_path = root / 'src' / 'componentes' / 'Inventario.tsx'

# Remove local barcode sample database object entirely
pattern_db = r"const baseDatosCodigosBarras = \{[\s\S]*?\};\s*useEffect\(\(\) => \{""
replacement_db = "const baseDatosCodigosBarras = {};\n\n  useEffect(() => {"
replace_in_file(escaner_path, pattern_db, replacement_db, flags=re.MULTILINE)

# Remove prueba code generation and button state
pattern_prueba_fn = r"const generarCodigoPrueba = \(\) => \{[\s\S]*?\};\n\n  if \(!permisosCamara\) \{""
replacement_prueba_fn = "const generarCodigoPrueba = () => {\n    // Ya no usamos códigos de prueba locales.\n    console.log('Eliminar función de prueba. Usa el escáner o ingresa el código manualmente.');\n  };\n\n  if (!permisosCamara) {"
replace_in_file(escaner_path, pattern_prueba_fn, replacement_prueba_fn, flags=re.MULTILINE)

# Remove prueba button from UI and local stats display
pattern_prueba_button = r"\s*<button className=\"button success\" onClick=\{generarCodigoPrueba\}>[\s\S]*?🎲 Código de Prueba</button>\s*"
replace_in_file(escaner_path, pattern_prueba_button, "")

pattern_stats = r"<p>[\s\S]*?<strong>Códigos de prueba locales:</strong>\{\}.*?</p>\s*"
replace_in_file(escaner_path, pattern_stats, "")

# Adjust buscarProductoEnAPI to call onProductoNoEncontrado when not found
pattern_api_not_found = r"else \{\s*console\.log\("❌ Producto no encontrado en Open Food Facts"\);\s*\}\s*\} catch \(error\) \{"
replacement_api_not_found = "else {\n        console.log(\"❌ Producto no encontrado en Open Food Facts\");\n        setProductoEncontrado(null);\n        if (onProductoNoEncontrado) {\n          onProductoNoEncontrado(codigo);\n        }\n      }\n    } catch (error) {"
replace_in_file(escaner_path, pattern_api_not_found, replacement_api_not_found, flags=re.MULTILINE)

# Also remove local test counter from stats footer
pattern_local_stats = r"<p>\s*<strong>Códigos de prueba locales:</strong> \{Object\.keys\(baseDatosCodigosBarras\)\.length\}</p>\s*"
replace_in_file(escaner_path, pattern_local_stats, "")

print('Edits completed for scanner component.')


SyntaxError: unterminated string literal (detected at line 18) (2861527192.py, line 18)